# M0 · 02 — Indexing & slicing

Slicing is how the GPT grabs *parts* of a tensor: `idx[:, -block_size:]`,
`self.tril[:T, :T]`, `logits[:, -1, :]`. It's the same idea as Python list
slicing, extended to multiple axes.

Key notation: **`start:stop:step`**, `stop` is *exclusive*, negatives count
from the end. A bare `:` means "all of this axis".

In [1]:
import torch
from m0_checks import check, check_tensor, TODO
torch.manual_seed(0)

In [2]:
# A little 1-D tensor to practise on:
x = torch.arange(10)   # tensor([0,1,2,3,4,5,6,7,8,9])
x

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

## 1. `:T` — the first T elements

Grab the **first 3** elements of `x` (`[0,1,2]`).

In [6]:
first3 = x[:3]
first3

tensor([0, 1, 2])

In [7]:
check('first three', first3, torch.tensor([0, 1, 2]))

✅ first three


True

## 2. Negative index — the last element

`-1` is the last position. Get the **last** element of `x`.

In [8]:
last = x[-1]
last

tensor(9)

In [9]:
check('last element', last, torch.tensor(9))

✅ last element


True

## 3. Last k elements — `x[-k:]`

This is exactly the GPT's context crop `idx[:, -block_size:]`.
Get the **last 4** elements of `x` (`[6,7,8,9]`).

In [10]:
last4 = x[-4:]
last4

tensor([6, 7, 8, 9])

In [11]:
check('last four', last4, torch.tensor([6, 7, 8, 9]))

✅ last four


True

## 4. 2-D slicing — top-left block

With two axes you slice each one, comma-separated: `m[rows, cols]`.
This is how `self.tril[:T, :T]` takes the top-left `T×T` corner of the mask.

In [12]:
m = torch.arange(16).reshape(4, 4)
m

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]])

Take the top-left **2×2** block of `m` (`[[0,1],[4,5]]`).

In [13]:
topleft = m[:2,:2]
topleft

tensor([[0, 1],
        [4, 5]])

In [14]:
check('top-left 2x2', topleft, torch.tensor([[0, 1], [4, 5]]))

✅ top-left 2x2


True

## 5. `:` means the whole axis — grab a column

`m[:, j]` keeps **all rows** but only column `j`. The GPT uses `logits[:, -1, :]`:
all batches, the **last** time step, all vocab.

Get the **last column** of `m` (`[3, 7, 11, 15]`).

In [15]:
last_col = m[:,-1]
last_col

tensor([ 3,  7, 11, 15])

In [16]:
check('last column', last_col, torch.tensor([3, 7, 11, 15]))

✅ last column


True

## 6. Putting it together — the GPT's `logits[:, -1, :]`

Given a fake `logits` of shape `(B, T, V)`, select the last time step for every
batch, keeping all V — result shape `(B, V)`.

In [17]:
logits = torch.arange(2 * 3 * 4).reshape(2, 3, 4)  # (B=2, T=3, V=4)
last_step = logits[:,-1,:]
last_step

tensor([[ 8,  9, 10, 11],
        [20, 21, 22, 23]])

In [18]:
check('last-step values', last_step, torch.tensor([[8, 9, 10, 11], [20, 21, 22, 23]]))
check_tensor('last-step shape (B,V)', last_step, shape=(2, 4))

✅ last-step values
✅ last-step shape (B,V)


True

## ✅ Recap

- `:` = whole axis; `start:stop` (stop exclusive); `-1` = last; `-k:` = last k.
- Multi-axis: `m[:2, :2]` = top-left block (that's `tril[:T,:T]`).
- `logits[:, -1, :]` = last time step per batch → shape `(B, V)`.

Next: **03 — shapes, reshape/view, transpose**.